# **Assignment #2 - Documents Search Engine**

### Problem Statement
Suppose you have a set of documents, each of which contains a set of statements. Write a program that reads the main concepts in the documents and constructs an index table (an n*m matrix A in which A[i][j] contains 1 if concept i is located in document j, 0 otherwise). The user is prompted to enter some compound statement, and the program displays the set of documents that fulfills the statement.
_______________


The program described below implements a document search engine. It ingests a set of documents, tokenizes them to extract key concept terms, and constructs a term-document matrix using `sklearn.CountVectorizer`. The user can then submit a logical search query, which is parsed and evaluated against the matrix to identify matching documents.

The program has three main modules:

1- **Document Reader and Tokenizer:** Constructs the term-document matrix.<br>
2- **Simple Parser:** Converts input sentences into propositional logic Abstract Syntax Trees (ASTs).<br>
3- **Document Search Engine (Reasoning Module):** return all documents that satisfy the query.<br><br>

The following figure illustrates the main components of the program and how they interact.

![Documents Search System Modules](documents_search.jpeg)

In [ ]:
import sys
!{sys.executable} -m pip install --user scikit-learn
!{sys.executable} -m pip install --user pandas

In [2]:
from IPython.display import Markdown
import pandas as pd
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

## **Module-1: Documents Reader & Tokenizer**

The following function reads a set of documents (*.txt files) that exists in a specific directory<br>

In [3]:
def read_documents(dir_path):
    # Reads all .txt files into a single DataFrame
    df = pd.DataFrame([{
        "filename": f.name,
        "content": f.read_text(encoding="utf-8")
        } for f in Path(dir_path).glob("*.txt")])
    
    print(f"{len(df)} documents were loaded successfully.")
    
    return df


The following function uses `CountVectorizer` to tokenize and transform the loaded documents into term-document matrix <br>

In [4]:
def get_term_document_matrix(documents_df):
    vectorizer = CountVectorizer(binary=True, stop_words="english", max_features=5000)
    term_doc_matrix = vectorizer.fit_transform(documents_df["content"])

    print(f"{(term_doc_matrix.shape[1])} terms were tokenized from documents content.")

    return pd.DataFrame(
        term_doc_matrix.T.toarray(),
        index=vectorizer.get_feature_names_out(),
        columns=documents_df["filename"]
    )

## **Module-2: Propositional Logic Parser**
The following python class implements a **simple** text parser that convert human sentences into propositional logic sentences
- The **input** for this parser is: human recognized raw sentences.<br>
<pre>
    eg. if today is holiday, then our services will not be available.
</pre>
- The **output** is a propositional logic abstract syntax tree representation of that sentence.<br>
<pre>
    eg.     ['IF', Q, ['NOT', P]]
            where: 
            { Q: 'today is holiday', P: 'Our services are available' }
</pre> 
<br/>

It first tokenizes and cleans the sentence using the third-party <a href="https://spacy.io/">`spaCy.nlp`</a> library, then uses the resulting tokens to build the AST.

**The allowed input sentence structure is illustrated below:**

```<br>
<Expression>    ::= <Iff_Expr>                            <---- Lowest Precedence

<Iff_Expr>      ::= <If_Expr> "IFF" <Iff_Expr> 
                    | <If_Expr>

<If_Expr>       ::= "IF" <Or_Expr> "THEN" <If_Expr> 
                    | <Or_Expr>

<Or_Expr>       ::= <And_Expr> "OR" <Or_Expr> 
                    | <And_Expr>

<And_Expr>      ::= <Not_Expr> "AND" <And_Expr> 
                    | <Not_Expr>

<Not_Expr>      ::= "NOT" <Not_Expr> 
                    | <Primary>

<Primary>       ::= "(" <Expression> ")" 
                    | <Atomic_Expr>

<Atomic_Expr>   ::= "Simple Atomic Human Sentence"          <---- Highest Precedence
```
<br/>


**Note:** All punctuation marks (e.g. ., !, ?, ,, []) are ignored and removed during processing, **except parentheses () which are kept for logical grouping.**.


In [ ]:
import sys
!{sys.executable} -m pip install --user -U pip
!{sys.executable} -m pip install --user spacy
!{sys.executable} -m spacy download en_core_web_sm

In [6]:
import spacy

class LogicParser:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.var_map = {}
        self.var_counter = ord('P')
        self.logical_ops = {"if", "then", "and", "or", "iff"}

    def _get_variable(self, prop_str):
        for var, text in self.var_map.items():
            if text == prop_str:
                return var
        var = chr(self.var_counter)
        self.var_map[var] = prop_str
        self.var_counter += 1
        return var

    def tokenize_text(self, text):
        text = text.lower().replace("if and only if", "iff")
        doc = self.nlp(text)
        tokens = []
        
        current_prop = []
        is_negated = False
        
        # Helper to flush words into a variable
        def flush_prop():
            nonlocal current_prop, is_negated
            if current_prop:
                prop_str = " ".join(current_prop).strip()
                var = self._get_variable(prop_str)
                if is_negated:
                    tokens.extend(["NOT", var])
                else:
                    tokens.append(var)
                current_prop = []
                is_negated = False
            else:
                if is_negated:
                    tokens.extend(["NOT"])
                    is_negated = False

        for token in doc:
            word = token.text
            
            # Preserve parentheses, skip other punctuation
            if token.is_punct and word not in ["(", ")"]:
                # True if the token is a punctuation mark (such as periods, commas, exclamation marks, brackets, or quotes) and False otherwise.
                continue

            if word in self.logical_ops or word in ["(", ")"]:
                flush_prop()
                tokens.append(word.upper())

            # token.dep_ returns the string label of the token's syntactic dependency relation to its head (parent word) in the sentence tree, as determined by spaCy's dependency parser.    
            elif token.dep_ == "neg" or word in ["no", "not"]:
                is_negated = True
            
            # Boundary 3: Skip all auxiliary verbs (will, can, would, did, have)
            elif token.dep_ in ["aux", "auxpass"]:
                continue
            else:
                # token.pos_ returns the string label for the token's Part-of-Speech (POS) tag based on the Universal Dependencies schema (e.g., NOUN, VERB, ADJ).
                if token.pos_ == "VERB":
                    # returns the string representation of the token's lemma, which is its base, dictionary, or canonical form.
                    current_prop.append(token.lemma_)
                else:
                    current_prop.append(word)
                    
        flush_prop()
        return tokens

    # Helper to find the root logical operator index in the tokens array
    def _find_top_level_op(self, tokens, op_list):
        level = 0
        for i, token in enumerate(tokens):
            if token == '(':
                level += 1
            elif token == ')':
                level -= 1
            elif level == 0 and token in op_list:
                return i
        return -1

    # Helper to remove redundant outer parentheses
    def _strip_outer_parens(self, tokens):
        while len(tokens) >= 2 and tokens[0] == '(' and tokens[-1] == ')':
            level = 0
            match = True
            # Check if the first '(' actually closes at the very end
            for token in tokens[:-1]:
                if token == '(': level += 1
                elif token == ')': level -= 1
                if level == 0: 
                    match = False # It closed early, so outer parens don't match
                    break
            
            if match:
                tokens = tokens[1:-1]
            else:
                break
        return tokens

    #Helper to ensure all parentheses open and close correctly.
    def validate_parens(self, tokens):
        level = 0
        for token in tokens:
            if token == '(': level += 1
            elif token == ')': level -= 1
            if level < 0:
                raise ValueError("Syntax Error: Closing parenthesis without a matching opening parenthesis.")
        if level > 0:
            raise ValueError("Syntax Error: Missing closing parenthesis.")

    def parse_ast(self, tokens):
        if not tokens:
            raise ValueError("Syntax Error: Missing expected proposition or operator.")
            
        tokens = self._strip_outer_parens(tokens)
        
        if len(tokens) == 1:
            if tokens[0] in ['IF', 'THEN', 'AND', 'OR', 'IFF', 'NOT']:
                raise ValueError(f"Syntax Error: singular operator '{tokens[0]}'.")
            return tokens[0]
            
        # Parse IFF 
        idx = self._find_top_level_op(tokens, ['IFF'])
        if idx != -1:
            if idx == 0 or idx == len(tokens) - 1:
                raise ValueError("Syntax Error: 'IFF' is missing a left or right proposition.")
            return ['IFF', self.parse_ast(tokens[:idx]), self.parse_ast(tokens[idx+1:])]
            
        # 3. Parse IF-THEN 
        if tokens[0] == 'IF':
            idx = self._find_top_level_op(tokens, ['THEN'])
            if idx == -1:
                raise ValueError("Syntax Error: 'IF' condition is missing a corresponding 'THEN'.")
            if idx == 1 or idx == len(tokens) - 1:
                raise ValueError("Syntax Error: 'IF...THEN' block is missing a condition or consequent.")
            return ['IF', self.parse_ast(tokens[1:idx]), self.parse_ast(tokens[idx+1:])]
            
        # 4. Parse OR 
        idx = self._find_top_level_op(tokens, ['OR'])
        if idx != -1:
            if idx == 0 or idx == len(tokens) - 1:
                raise ValueError("Syntax Error: 'OR' is missing a left or right proposition.")
            return ['OR', self.parse_ast(tokens[:idx]), self.parse_ast(tokens[idx+1:])]
            
        # 5. Parse AND 
        idx = self._find_top_level_op(tokens, ['AND'])
        if idx != -1:
            if idx == 0 or idx == len(tokens) - 1:
                raise ValueError("Syntax Error: 'AND' is missing a left or right proposition.")
            return ['AND', self.parse_ast(tokens[:idx]), self.parse_ast(tokens[idx+1:])]
            
        # 6. Parse NOT
        if tokens[0] == 'NOT':
            if len(tokens) == 1:
                raise ValueError("Syntax Error: 'NOT' is missing a target proposition.")
            return ['NOT', self.parse_ast(tokens[1:])]
            
        # 7. Catch-all for unparsed garbage (e.g., "P Q" without an operator between them)
        raise ValueError(f"Syntax Error: Invalid sequence of tokens: {tokens}. Missing logical operator between propositions.")

    def convert(self, text):
        flat_tokens = self.tokenize_text(text)
        # check parentheses before parsing
        self.validate_parens(flat_tokens)
        return self.parse_ast(flat_tokens)

## **Module-3: Documents Search Engine**

This is the main module of the program. It takes the outputs of the previous two modules (the document tokenizer and the propositional logic parser) and performs logical reasoning to check which documents match the user's query. Specifically, it evaluates the truth value of the query by treating each document column as a logical model. If a column evaluates to True, the document it represents satisfies the query.

The function returns a filtered two-dimensional DataFrame:
- `Rows`: Keeps only the rows corresponding to the search terms.
- `Columns`: Keeps only the documents that pass the boolean filter.


In [48]:
def search(query, tdm, symbols_map):

    def documents_filter(col):
        return pl_true(query, col, symbols_map)

    symbols = list(symbols_map.values())
    
    missing_keys = list(set(symbols).difference(tdm.index))
    if missing_keys:
        print(f"\nError: The following terms are missing from the `term-document` DataFrame: {missing_keys}")
        # return empty dataframe
        return pd.DataFrame()
    else:
        documents_mask = tdm.apply(documents_filter, axis=0)
        return tdm.loc[tdm.index.isin(symbols), documents_mask]


def pl_true(query, doc_col, symbols_map):
    "Return True if query is true in doc_col model, else False."
    if isinstance(query, str): 
        return doc_col[symbols_map[query]] == 1
    op, args = query[0], query[1:]
    if op == 'NOT': 
        return not pl_true(args[0], doc_col, symbols_map)
    elif op == 'AND': 
        return all(pl_true(arg, doc_col, symbols_map) for arg in args)
    elif op == 'OR': 
        return any(pl_true(arg, doc_col, symbols_map) for arg in args)
    elif op == 'IF': 
        return (not pl_true(args[0], doc_col, symbols_map)) or pl_true(args[1], doc_col, symbols_map)
    elif op == 'IFF': 
        return pl_true(args[0], doc_col, symbols_map) == pl_true(args[1], doc_col, symbols_map)
    else: print("op", op, f"Error: {op} not recognized." )


----

### Testing the Complete Pipeline Using Sample Documents Folder:

* Load Documents:

In [49]:
test_dir = "./documents"
test_documents_df = read_documents(test_dir)

display(test_documents_df.head(10))

4 documents were loaded successfully.


,filename,content
0,doc1.txt,"spark, rdd, spark\nData science involves stati..."
1,doc2.txt,"rdd, cache\nMachine learning relies on algorit..."
2,doc3.txt,"spark, cluster\nAlgorithms and data structures..."
3,doc4.txt,"spark, cluster\nAlgorithms and data structures..."


* Tokenize Documents:

In [50]:
test_tdm = get_term_document_matrix(test_documents_df)
test_tdm

15 terms were tokenized from documents content.


filename,doc1.txt,doc2.txt,doc3.txt,doc4.txt
algorithms,0,1,1,1
cache,0,1,0,0
cluster,0,0,1,1
computer,0,0,1,0
data,1,0,1,1
fundamental,0,0,1,0
involves,1,0,0,0
learning,1,1,0,1
machine,1,1,0,1
rdd,1,1,0,0


* Parse the user query sentence and perform document search:

In [52]:
queries = ["science and spark", "machine and learning or data", "if data then structures", "rdd"]

for query in queries:
    test_parser = LogicParser()
    test_query_ast =  test_parser.convert(query)
    test_results = search(test_parser.convert(query), test_tdm, test_parser.var_map)
    print(f"\n- {query}")
    print(" , ".join(test_results.columns))


- science and spark
doc1.txt , doc3.txt

- machine and learning or data
doc1.txt , doc2.txt , doc3.txt , doc4.txt

- if data then structures
doc2.txt , doc3.txt , doc4.txt

- rdd
doc1.txt , doc2.txt


______________________________________________________________________

# **Run Documents Search Engine:**
### ▶ Enter the path of your documents directory by running the following cell:

In [46]:
document_dir = input("Please provide a valid directory path that contains the documents:")

if(document_dir):
    print(f"\nDocuments Directory:\t{document_dir}")
    parsedKB = []
    documents_df = read_documents(document_dir)
    tdm = get_term_document_matrix(documents_df)

tdm   


Documents Directory:	articles
5 documents were loaded successfully.
342 terms were tokenized from documents content.


filename,ai.txt,big-data.txt,knowledge-representation.txt,machine-learning.txt,nlp.txt
accuracy,0,1,0,1,0
achieve,0,0,1,1,0
actionable,0,0,1,0,1
acts,0,1,0,0,0
actually,1,0,0,0,0
...,...,...,...,...,...
virtual,0,0,0,0,1
volume,0,1,0,0,0
way,0,0,0,0,1
words,0,0,0,0,1


### ▶ Enter your search query by **Re-running** the following cell:

In [57]:
query = input("Please enter your query:")

if(query):
    parser  = LogicParser()
    results = search(parser.convert(query), tdm, parser.var_map)
    display(results)
    if(len(results.columns) > 0):
        display(Markdown(f"## Your query '{query}' is satisfied in:\t"))
        for document in results.columns:
            display(Markdown(f"### - {document}"))
    else: 
        display(Markdown(f"## Your query '{query}' is not satisfied in any document."))

filename,big-data.txt,machine-learning.txt
accuracy,1,1
data,1,1


## Your query 'data and accuracy' is satisfied in:	

### - big-data.txt

### - machine-learning.txt